<div style="background-color:#2E7D32; padding:20px; border-radius:10px; text-align:center;">
    <h1 style="color:white; margin:0;">
        Marketing Campaign Success Estimation Model
    </h1>
</div>

## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import export_text

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

## Read data

In [24]:
data = pd.read_csv(r'marketing.csv')

In [25]:
data.head()

,ID,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,response,result
0,13829,29,technician,single,tertiary,no,18254,no,no,cellular,11,may,2,-1,0,unknown,no
1,22677,26,services,single,secondary,no,512,yes,yes,unknown,5,jun,3,-1,0,unknown,no
2,10541,30,management,single,secondary,no,135,no,no,cellular,14,aug,2,-1,0,unknown,no
3,13689,41,technician,married,unknown,no,30,yes,no,cellular,10,jul,1,-1,0,unknown,no
4,11304,27,admin.,single,secondary,no,321,no,yes,unknown,2,sep,1,-1,0,unknown,no


| Column | Meaning |
|---|---|
| **ID** | Unique identification number of the customer |
| **age** | Age of the customer |
| **job** | Type of job or occupation of the customer |
| **marital** | Marital status of the customer |
| **education** | Education level of the customer |
| **default** | Whether the customer has credit in default |
| **balance** | Average yearly balance of the customer |
| **housing** | Whether the customer has a housing loan |
| **loan** | Whether the customer has a personal loan |
| **contact** | Communication method used to contact the customer |
| **day** | Day of the month when the customer was contacted |
| **month** | Month when the customer was contacted |
| **campaign** | Number of contacts performed during the current marketing campaign |
| **pdays** | Number of days since the customer was last contacted in a previous campaign; **-1 means the customer was not previously contacted** |
| **previous** | Number of contacts performed before the current marketing campaign |
| **response** | Result of the previous marketing campaign |
| **result** | Outcome of the current marketing campaign / target variable |

# Common Pre-processing steps

## Descriptive statistics

In [26]:
pd.set_option('display.max_columns', None)

data.describe(include='all')

,ID,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,response,result
count,12870.000000,12870.000000,12870,12870,12870,12870,12870.000000,12870,12870,12870,12870.000000,12870,12870.000000,12870.000000,12870.000000,12870,12870
unique,NaN,NaN,12,3,4,2,NaN,2,2,3,NaN,12,NaN,NaN,NaN,4,2
top,NaN,NaN,management,married,secondary,no,NaN,yes,no,cellular,NaN,may,NaN,NaN,NaN,unknown,no
freq,NaN,NaN,2858,7490,6368,12662,NaN,6605,11060,8756,NaN,3594,NaN,NaN,NaN,10070,8903
mean,16434.500000,41.091142,NaN,NaN,NaN,NaN,1483.774437,NaN,NaN,NaN,15.641103,NaN,2.659130,45.555478,0.688967,NaN,NaN
std,3715.393317,11.305560,NaN,NaN,NaN,NaN,3311.055181,NaN,NaN,NaN,8.368983,NaN,2.863507,104.449411,2.049696,NaN,NaN
min,10000.000000,18.000000,NaN,NaN,NaN,NaN,-6847.000000,NaN,NaN,NaN,1.000000,NaN,1.000000,-1.000000,0.000000,NaN,NaN
25%,13217.250000,32.000000,NaN,NaN,NaN,NaN,102.000000,NaN,NaN,NaN,8.000000,NaN,1.000000,-1.000000,0.000000,NaN,NaN
50%,16434.500000,39.000000,NaN,NaN,NaN,NaN,515.000000,NaN,NaN,NaN,16.000000,NaN,2.000000,-1.000000,0.000000,NaN,NaN
75%,19651.750000,49.000000,NaN,NaN,NaN,NaN,1591.750000,NaN,NaN,NaN,21.000000,NaN,3.000000,-1.000000,0.000000,NaN,NaN


In [16]:
data.dtypes

ID            int64
age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
campaign      int64
pdays         int64
previous      int64
response     object
result       object
dtype: object

## Drop unnecessary columns

In [17]:
data['pdays'].value_counts()

pdays
-1      10067
 182       76
 92        75
 91        65
 181       63
        ...  
 215        1
 557        1
 54         1
 680        1
 427        1
Name: count, Length: 461, dtype: int64

In [18]:
data['previous'].value_counts()

previous
0     10067
1       921
2       692
3       391
4       248
5       182
6       109
7        60
8        54
9        28
10       28
12       20
11       18
13       11
17        6
14        6
20        4
15        4
16        4
23        2
21        2
18        2
19        2
29        2
30        2
37        1
58        1
22        1
26        1
55        1
Name: count, dtype: int64

In [19]:
data.nunique()

ID           12870
age             76
job             12
marital          3
education        4
default          2
balance       4188
housing          2
loan             2
contact          3
day             31
month           12
campaign        36
pdays          461
previous        30
response         4
result           2
dtype: int64

In [28]:
data.drop(data[['ID']],axis=1, inplace=True)

In [29]:
data.drop(data[['previous']],axis=1, inplace=True)

In [30]:
data.drop(data[['pdays']],axis=1, inplace=True)

## Convert 'result' column to numeric. yes=0, no=1

In [31]:
data['result'] = data['result'].map({'yes': 0, 'no': 1})

In [32]:
data['result'].value_counts()

result
1    8903
0    3967
Name: count, dtype: int64

## Check missing values

In [33]:
data.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
campaign     0
response     0
result       0
dtype: int64

In [49]:
# create 2 same dataframes in order to proced individual pre porocessing steps as below:

In [34]:
data_lr = data.copy()
data_rf = data.copy()

In [51]:
# Declare target
# Declare inputs_lr (from data_lr) and inputs_rf (from data_rf)

In [35]:
inputs_lr = data_lr.drop(columns=['result'])
inputs_rf = data_rf.drop(columns=['result'])

In [36]:
target = data['result']

In [54]:
# Split the data into train and test, consider inputs for LogReg and RF

In [37]:
X_train_lr, X_test_lr, y_train, y_test = train_test_split(inputs_lr, target, test_size=0.2, random_state=42)
X_train_rf, X_test_rf, y_train, y_test = train_test_split(inputs_rf, target, test_size=0.2, random_state=42)

In [38]:
print('X_train_lr shape', X_train_lr.shape)
print('X_test_lr shape', X_test_lr.shape)

print('X_train_rf shape', X_train_rf.shape)
print('X_test_rf shape', X_test_rf.shape)

print('y_train shape', y_train.shape)
print('y_test shape', y_test.shape)

X_train_lr shape (10296, 13)
X_test_lr shape (2574, 13)
X_train_rf shape (10296, 13)
X_test_rf shape (2574, 13)
y_train shape (10296,)
y_test shape (2574,)


```mermaid
flowchart TD
    A["📊 RAW DATA"] --> B["🔵 Logistic Regression"]
    A --> C["🟢 Random Forest"]

    B --> B1["📦 Binning"]
    B1 --> B2["🔄 WOE"]
    B2 --> B3["📈 IV Analysis"]
    B3 --> B4["🔗 Intercorrelation"]
    B4 --> B5["🎯 Logistic Regression"]

    C --> C1["🔢 Encoding"]
    C1 --> C2["🏷️ LabelEncoder"]
    C2 --> C3["🌲 RF Feature Selection"]
    C3 --> C4["🎯 Random Forest"]

    style A fill:#6C5CE7,color:#fff,stroke:#4834D4,stroke-width:3px

    style B fill:#74B9FF,color:#fff,stroke:#0984E3,stroke-width:2px
    style B1 fill:#EAF6FF,stroke:#74B9FF
    style B2 fill:#EAF6FF,stroke:#74B9FF
    style B3 fill:#FFF4CC,stroke:#FDCB6E
    style B4 fill:#FFF4CC,stroke:#FDCB6E
    style B5 fill:#0984E3,color:#fff,stroke:#0652DD,stroke-width:3px

    style C fill:#55EFC4,color:#075E54,stroke:#00B894,stroke-width:2px
    style C1 fill:#E5FFF7,stroke:#55EFC4
    style C2 fill:#E5FFF7,stroke:#55EFC4
    style C3 fill:#FFF4CC,stroke:#FDCB6E
    style C4 fill:#00B894,color:#fff,stroke:#00896A,stroke-width:3px
```

# Pre processing steps for Logistic Regression

In [57]:
# Convert into woe all features if X_train_lr

In [40]:
numeric_columns = X_train_lr.select_dtypes(include="number").columns
display(numeric_columns)

categorical_columns = X_train_lr.select_dtypes(include=["object", "category"]).columns
display(categorical_columns)

Index(['age', 'balance', 'day', 'campaign'], dtype='object')

Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'response'],
      dtype='object')

```mermaid
flowchart TD

    A["📊 X_train_lr"]:::data

    A --> B["🔢 NUMERIC FEATURES"]:::numeric
    A --> C["🔤 CATEGORICAL FEATURES"]:::categorical

    B --> B1["age<br/>balance<br/>campaign<br/>day<br/>pdays<br/>previous"]:::features
    C --> C1["job<br/>marital<br/>education<br/>default<br/>housing<br/>loan<br/>contact<br/>month<br/>response"]:::features

    B1 --> D["📦 BINNING"]:::process
    C1 --> E["🔄 WOE ENCODING"]:::process

    D --> F["WOE"]:::woe
    E --> G["WOE"]:::woe

    F --> H["🎯 WOE-TRANSFORMED X"]:::result
    G --> H

    H --> I["🤖 LOGISTIC REGRESSION"]:::model
    I --> J["📈 PREDICTION"]:::prediction

    J --> K["YES (0)"]:::yes
    J --> L["NO (1)"]:::no


    classDef data fill:#6C5CE7,stroke:#4834D4,color:#fff,stroke-width:3px;
    classDef numeric fill:#74B9FF,stroke:#0984E3,color:#fff,stroke-width:2px;
    classDef categorical fill:#55EFC4,stroke:#00B894,color:#075E54,stroke-width:2px;
    classDef features fill:#F8F9FA,stroke:#B2BEC3,color:#2D3436;
    classDef process fill:#FFEAA7,stroke:#FDCB6E,color:#2D3436,stroke-width:2px;
    classDef woe fill:#A29BFE,stroke:#6C5CE7,color:#fff,stroke-width:2px;
    classDef result fill:#DFF9FB,stroke:#22A6B3,color:#2D3436,stroke-width:2px;
    classDef model fill:#0984E3,stroke:#0652DD,color:#fff,stroke-width:3px;
    classDef prediction fill:#FD79A8,stroke:#E84393,color:#fff,stroke-width:2px;
    classDef yes fill:#00B894,stroke:#00896A,color:#fff,stroke-width:2px;
    classDef no fill:#FF7675,stroke:#D63031,color:#fff,stroke-width:2px;
```

In [41]:
woe_maps = {}
bin_maps = {}

X_train_lr_woe = X_train_lr.copy()
X_train_lr_woe['target'] = y_train.values

features = X_train_lr.columns
display(features)

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'campaign', 'response'],
      dtype='object')

In [42]:
for var in features:

    if X_train_lr[var].dtype != object:

        q1 = X_train_lr[var].quantile(0.25)
        q2 = X_train_lr[var].quantile(0.50)
        q3 = X_train_lr[var].quantile(0.75)

        bins = [-np.inf, q1, q2, q3, np.inf]

        train_bin = pd.cut(X_train_lr[var],bins=bins,duplicates='drop')
        # duplicates='drop' is used because quantile values can sometimes be the same.

        grouped = X_train_lr_woe.groupby([train_bin, 'target'], observed=False)['target'].count().unstack()

        grouped['woe'] = np.log((grouped[0] / grouped[0].sum()) / (grouped[1] / grouped[1].sum()))

        woe_maps[var] = grouped['woe']
        bin_maps[var] = bins

        X_train_lr_woe[var + '_woe'] = train_bin.map(grouped['woe'])

    else:

        grouped = X_train_lr_woe.groupby([var, 'target'])['target'].count().unstack()

        grouped['woe'] = np.log((grouped[0] / grouped[0].sum()) / (grouped[1] / grouped[1].sum()))

        woe_maps[var] = grouped['woe']

        X_train_lr_woe[var + '_woe'] = X_train_lr[var].map(grouped['woe'])

In [44]:
X_train_lr_woe.head(3)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,response,target,age_woe,job_woe,marital_woe,education_woe,default_woe,balance_woe,housing_woe,loan_woe,contact_woe,day_woe,month_woe,campaign_woe,response_woe
7529,40,technician,divorced,tertiary,no,2611,yes,no,cellular,22,jul,2,unknown,1,-0.301707,-0.127224,0.079507,0.295142,0.010059,0.393440,-0.468950,0.077349,0.281322,0.015188,-0.257731,-0.017526,-0.267003
10042,34,blue-collar,married,secondary,no,932,no,yes,cellular,18,nov,2,other,1,-0.102611,-0.602644,-0.170011,-0.122578,0.010059,0.064435,0.414527,-0.546299,0.281322,-0.262208,-0.089767,-0.017526,0.434486
9396,57,retired,married,unknown,no,-157,no,no,cellular,28,aug,9,unknown,1,0.207856,0.921662,-0.170011,0.122904,0.010059,-0.491948,0.414527,0.077349,0.281322,0.015188,-0.119167,-0.532487,-0.267003


In [45]:
woe_maps

{'age': age
 (-inf, 32.0]    0.181321
 (32.0, 39.0]   -0.102611
 (39.0, 49.0]   -0.301707
 (49.0, inf]     0.207856
 Name: woe, dtype: float64,
 'job': job
 admin.          -0.003843
 blue-collar     -0.602644
 entrepreneur    -0.262182
 housemaid       -0.174280
 management       0.167020
 retired          0.921662
 self-employed    0.095450
 services        -0.162756
 student          1.142569
 technician      -0.127224
 unemployed       0.309066
 unknown          0.053088
 Name: woe, dtype: float64,
 'marital': marital
 divorced    0.079507
 married    -0.170011
 single      0.269317
 Name: woe, dtype: float64,
 'education': education
 primary     -0.314730
 secondary   -0.122578
 tertiary     0.295142
 unknown      0.122904
 Name: woe, dtype: float64,
 'default': default
 no     0.010059
 yes   -0.778942
 Name: woe, dtype: float64,
 'balance': balance
 (-inf, 106.0]     -0.491948
 (106.0, 519.0]    -0.041606
 (519.0, 1600.0]    0.064435
 (1600.0, inf]      0.393440
 Name: woe, dtyp

In [ ]:
# Transform woe values to X_test_lr

In [46]:
X_test_lr_woe = X_test_lr.copy()

for var in features:

    if X_train_lr[var].dtype != object:

        test_bin = pd.cut(X_test_lr[var],bins=bin_maps[var],duplicates='drop')

        X_test_lr_woe[var + '_woe'] = test_bin.map(woe_maps[var])

    else:

        X_test_lr_woe[var + '_woe'] = X_test_lr[var].map(woe_maps[var])
        
X_test_lr_woe

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,response,age_woe,job_woe,marital_woe,education_woe,default_woe,balance_woe,housing_woe,loan_woe,contact_woe,day_woe,month_woe,campaign_woe,response_woe
11527,56,services,married,secondary,no,705,yes,no,unknown,29,may,2,unknown,0.207856,-0.162756,-0.170011,-0.122578,0.010059,0.064435,-0.468950,0.077349,-1.121337,0.015188,-0.663454,-0.017526,-0.267003
8653,25,management,single,tertiary,no,808,no,no,cellular,18,sep,2,failure,0.181321,0.167020,0.269317,0.295142,0.010059,0.064435,0.414527,0.077349,0.281322,-0.262208,1.798277,-0.017526,0.143065
8,59,management,married,tertiary,no,3342,no,no,cellular,18,mar,2,other,0.207856,0.167020,-0.170011,0.295142,0.010059,0.393440,0.414527,0.077349,0.281322,-0.262208,2.064812,-0.017526,0.434486
839,31,management,single,tertiary,no,4247,no,no,cellular,18,aug,3,unknown,0.181321,0.167020,0.269317,0.295142,0.010059,0.393440,0.414527,0.077349,0.281322,-0.262208,-0.119167,-0.036715,-0.267003
2340,43,management,married,tertiary,no,3518,no,no,cellular,28,dec,2,success,-0.301707,0.167020,-0.170011,0.295142,0.010059,0.393440,0.414527,0.077349,0.281322,0.015188,1.837301,-0.017526,2.575610
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2464,39,management,married,secondary,no,1035,yes,no,cellular,20,apr,1,unknown,-0.102611,0.167020,-0.170011,-0.122578,0.010059,0.064435,-0.468950,0.077349,0.281322,-0.262208,0.696923,0.249309,-0.267003
4508,32,technician,single,secondary,no,4843,no,no,cellular,8,aug,4,unknown,0.181321,-0.127224,0.269317,-0.122578,0.010059,0.393440,0.414527,0.077349,0.281322,0.061331,-0.119167,-0.532487,-0.267003
11301,23,technician,single,secondary,no,283,yes,no,cellular,22,sep,1,success,0.181321,-0.127224,0.269317,-0.122578,0.010059,-0.041606,-0.468950,0.077349,0.281322,0.015188,1.798277,0.249309,2.575610
1805,45,admin.,married,secondary,no,236,no,no,cellular,20,aug,2,unknown,-0.301707,-0.003843,-0.170011,-0.122578,0.010059,-0.041606,0.414527,0.077349,0.281322,-0.262208,-0.119167,-0.017526,-0.267003


# Data Distribution

In [47]:
from scipy import stats


for i in X_train_lr_woe.columns:
    
    if X_train_lr_woe[i].dtype in ['int64', 'float64']:
        
        kstest_statistic, kstest_p_value = stats.kstest(X_train_lr_woe[i], 'norm')
        
        print(f'Column: {i}')
        print(f'Kolmogorov-Smirnov Test:')
        print(f'Test Statistic: {kstest_statistic}')
        print(f'p-value: {kstest_p_value}')
        
        if kstest_p_value > 0.05:
            print('Data looks normally distributed')
            print()
        else:
            print('Data does not look normally distributed')
            print()

Column: age
Kolmogorov-Smirnov Test:
Test Statistic: 1.0
p-value: 0.0
Data does not look normally distributed

Column: balance
Kolmogorov-Smirnov Test:
Test Statistic: 0.8489803272985952
p-value: 0.0
Data does not look normally distributed

Column: day
Kolmogorov-Smirnov Test:
Test Statistic: 0.9675373583393111
p-value: 0.0
Data does not look normally distributed

Column: campaign
Kolmogorov-Smirnov Test:
Test Statistic: 0.8413447460685429
p-value: 0.0
Data does not look normally distributed

Column: target
Kolmogorov-Smirnov Test:
Test Statistic: 0.5297674344912313
p-value: 0.0
Data does not look normally distributed

Column: job_woe
Kolmogorov-Smirnov Test:
Test Statistic: 0.31197933615578644
p-value: 0.0
Data does not look normally distributed

Column: marital_woe
Kolmogorov-Smirnov Test:
Test Statistic: 0.43250070650386246
p-value: 0.0
Data does not look normally distributed

Column: education_woe
Kolmogorov-Smirnov Test:
Test Statistic: 0.3839425837393152
p-value: 0.0
Data does no

### Check correlation (threshold > 70%)

In [48]:
def intercorrelation(data, threshold=0.7):
    
    woe_features = data.filter(regex=r'_woe$', axis=1).copy()
    
    corr_matrix = woe_features.corr(method='spearman', numeric_only=True)
    
    mask = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    
    high_corr = corr_matrix.where(mask).stack().reset_index()
    high_corr.columns = ["Variable 1", "Variable 2", "Correlation"]
    
    high_corr = high_corr[high_corr["Correlation"].abs() >= threshold]
    
    return high_corr.reset_index(drop=True)


intercorrelated_result = intercorrelation(X_train_lr_woe)
intercorrelated_result

,Variable 1,Variable 2,Correlation


### Select final _woe variables and assign woe data to X_train_lr_fin and X_test_lr_fin variables

In [49]:
fin_vars = X_train_lr_woe.filter(regex=r'_woe$', axis=1).columns.tolist()
fin_vars

['age_woe',
 'job_woe',
 'marital_woe',
 'education_woe',
 'default_woe',
 'balance_woe',
 'housing_woe',
 'loan_woe',
 'contact_woe',
 'day_woe',
 'month_woe',
 'campaign_woe',
 'response_woe']

In [50]:
X_train_lr_fin = X_train_lr_woe[fin_vars]
X_test_lr_fin = X_test_lr_woe[fin_vars]

# Pre processing steps for Random Forest 

### Using LabelEncoder convert variables of X_train_rf into numerical

In [51]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

for col in X_train_rf.select_dtypes(include='object').columns:
    le = LabelEncoder()
    
    X_train_rf[col] = le.fit_transform(X_train_rf[col].astype(str))
    
    label_encoders[col] = le

In [ ]:
# Transform those conversions to X_test_rf

In [52]:
X_test_rf = X_test_rf.copy()

for col, le in label_encoders.items():
    X_test_rf[col] = le.transform(X_test_rf[col].astype(str))

In [53]:
X_train_rf.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,response
7529,40,9,0,2,0,2611,1,0,0,22,5,2,3
10042,34,1,1,1,0,932,0,1,0,18,9,2,1
9396,57,5,1,3,0,-157,0,0,0,28,1,9,3
188,36,0,1,1,0,131,1,0,0,2,3,1,3
1123,23,1,2,0,0,3198,1,0,0,15,8,1,0


In [54]:
X_test_rf.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,response
11527,56,7,1,1,0,705,1,0,2,29,8,2,3
8653,25,4,2,2,0,808,0,0,0,18,11,2,0
8,59,4,1,2,0,3342,0,0,0,18,7,2,1
839,31,4,2,2,0,4247,0,0,0,18,1,3,3
2340,43,4,1,2,0,3518,0,0,0,28,2,2,2


In [55]:
X_train_rf.dtypes

age          int64
job          int64
marital      int64
education    int64
default      int64
balance      int64
housing      int64
loan         int64
contact      int64
day          int64
month        int64
campaign     int64
response     int64
dtype: object

In [56]:
X_test_rf.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,response
11527,56,7,1,1,0,705,1,0,2,29,8,2,3
8653,25,4,2,2,0,808,0,0,0,18,11,2,0
8,59,4,1,2,0,3342,0,0,0,18,7,2,1
839,31,4,2,2,0,4247,0,0,0,18,1,3,3
2340,43,4,1,2,0,3518,0,0,0,28,2,2,2


# Modeling Steps

In [ ]:
# create evaluate() function for classification use

In [57]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score

def evaluate(model, X_train, y_train, X_test, y_test):
    
    '''Predictions and probabilities for the training set'''
    
    y_train_pred = model.predict(X_train)
    y_train_prob = model.predict_proba(X_train)[:, 1]

    '''Predictions and probabilities for the test set'''
    
    y_test_pred = model.predict(X_test)
    y_test_prob = model.predict_proba(X_test)[:, 1]

    '''Calculate metrics for the training set''' 
    
    roc_train_prob = roc_auc_score(y_train, y_train_prob)
    gini_train_prob = roc_train_prob * 2 - 1
    precision_train = precision_score(y_train, y_train_pred)
    recall_train = recall_score(y_train, y_train_pred)

    '''Calculate metrics for the test set'''
    
    roc_test_prob = roc_auc_score(y_test, y_test_prob)
    gini_test_prob = roc_test_prob * 2 - 1
    precision_test = precision_score(y_test, y_test_pred)
    recall_test = recall_score(y_test, y_test_pred)

    results = pd.DataFrame({
        'Dataset': ['Train', 'Test'],
        'Gini': [gini_train_prob * 100, gini_test_prob * 100],
        'Precision': [precision_train, precision_test],
        'Recall': [recall_train, recall_test]
    })

    return results

## Default Models

In [ ]:
# Train Logistic Regression model and check performance

In [58]:
X_train_lr_woe

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,response,target,age_woe,job_woe,marital_woe,education_woe,default_woe,balance_woe,housing_woe,loan_woe,contact_woe,day_woe,month_woe,campaign_woe,response_woe
7529,40,technician,divorced,tertiary,no,2611,yes,no,cellular,22,jul,2,unknown,1,-0.301707,-0.127224,0.079507,0.295142,0.010059,0.393440,-0.468950,0.077349,0.281322,0.015188,-0.257731,-0.017526,-0.267003
10042,34,blue-collar,married,secondary,no,932,no,yes,cellular,18,nov,2,other,1,-0.102611,-0.602644,-0.170011,-0.122578,0.010059,0.064435,0.414527,-0.546299,0.281322,-0.262208,-0.089767,-0.017526,0.434486
9396,57,retired,married,unknown,no,-157,no,no,cellular,28,aug,9,unknown,1,0.207856,0.921662,-0.170011,0.122904,0.010059,-0.491948,0.414527,0.077349,0.281322,0.015188,-0.119167,-0.532487,-0.267003
188,36,admin.,married,secondary,no,131,yes,no,cellular,2,feb,1,unknown,1,-0.102611,-0.003843,-0.170011,-0.122578,0.010059,-0.041606,-0.468950,0.077349,0.281322,0.061331,0.360622,0.249309,-0.267003
1123,23,blue-collar,single,primary,no,3198,yes,no,cellular,15,may,1,failure,1,0.181321,-0.602644,0.269317,-0.314730,0.010059,0.393440,-0.468950,0.077349,0.281322,0.179206,-0.663454,0.249309,0.143065
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11964,44,management,married,secondary,no,2897,no,no,cellular,5,apr,1,other,1,-0.301707,0.167020,-0.170011,-0.122578,0.010059,0.393440,0.414527,0.077349,0.281322,0.061331,0.696923,0.249309,0.434486
5191,44,management,married,tertiary,no,1058,no,no,cellular,11,mar,3,unknown,0,-0.301707,0.167020,-0.170011,0.295142,0.010059,0.064435,0.414527,0.077349,0.281322,0.179206,2.064812,-0.036715,-0.267003
5390,29,blue-collar,married,primary,no,220,yes,no,unknown,4,jun,2,unknown,1,0.181321,-0.602644,-0.170011,-0.314730,0.010059,-0.041606,-0.468950,0.077349,-1.121337,0.061331,-0.097773,-0.017526,-0.267003
860,51,technician,married,secondary,no,317,no,no,cellular,13,aug,4,unknown,1,0.207856,-0.127224,-0.170011,-0.122578,0.010059,-0.041606,0.414527,0.077349,0.281322,0.179206,-0.119167,-0.532487,-0.267003


In [59]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_lr_fin, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [60]:
logistic_results = evaluate(
    logistic_model,
    X_train_lr_fin,
    y_train,
    X_test_lr_fin,
    y_test
)

logistic_results

,Dataset,Gini,Precision,Recall
0,Train,53.115347,0.770363,0.938064
1,Test,50.507083,0.786741,0.934986


In [ ]:
# Train Random Forest model and check performance

In [61]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf_model.fit(X_train_rf, y_train)

RandomForestClassifier(n_estimators=300, random_state=42)

In [62]:
random_forest_results = evaluate(
    rf_model,
    X_train_rf,
    y_train,
    X_test_rf,
    y_test
)

random_forest_results

,Dataset,Gini,Precision,Recall
0,Train,100.000000,1.000000,1.000000
1,Test,55.097435,0.809759,0.905234


### ⚠️ Overfitting

The Random Forest model shows clear signs of **overfitting**. While the training Gini is **100%**, the test Gini drops to **55.57%**, resulting in a large **44.43% gap**.

This indicates that the model has learned the training data too closely and does not generalize well to unseen data.

**Next step:** Feature importance analysis and hyperparameter tuning will be performed to reduce overfitting.


## Models with important features

In [ ]:
# For Logistic Regression

# Use Univariate Analysis to evaluate which variables are important
# Train and Test gini more than 10%, and gap between train test less than 5%
# Use survived variables to build new model and check performance

In [63]:
univariate_results = []

for feature in X_train_lr_fin.columns:
    
    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )
    
    # Train model using only one feature
    model.fit(X_train_lr_fin[[feature]], y_train)
    
    # Predictions probabilities
    train_prob = model.predict_proba(X_train_lr_fin[[feature]])[:, 1]
    test_prob = model.predict_proba(X_test_lr_fin[[feature]])[:, 1]
    
    # Calculate Gini
    train_auc = roc_auc_score(y_train, train_prob)
    test_auc = roc_auc_score(y_test, test_prob)
    
    train_gini = (2 * train_auc - 1) * 100
    test_gini = (2 * test_auc - 1) * 100
    
    gini_gap = abs(train_gini - test_gini)
    
    univariate_results.append({
        'Variable': feature,
        'Train Gini': train_gini,
        'Test Gini': test_gini,
        'Gini Gap': gini_gap
    })

univariate_results = pd.DataFrame(univariate_results)

univariate_results

,Variable,Train Gini,Test Gini,Gini Gap
0,age_woe,11.113599,9.294236,1.819363
1,job_woe,21.945498,15.705238,6.240260
2,marital_woe,10.232848,10.263323,0.030476
3,education_woe,11.784674,11.128243,0.656431
4,default_woe,0.992440,0.754581,0.237859
5,balance_woe,16.821131,16.527982,0.293149
6,housing_woe,21.653471,17.994969,3.658501
7,loan_woe,6.751920,8.702839,1.950919
8,contact_woe,21.886021,22.330818,0.444797
9,day_woe,8.382252,12.310602,3.928349


In [64]:
survived_vars = univariate_results[
    (univariate_results['Train Gini'] > 10) &
    (univariate_results['Test Gini'] > 10) &
    (univariate_results['Gini Gap'] < 5)
]['Variable'].tolist()

In [65]:
survived_vars

['marital_woe',
 'education_woe',
 'balance_woe',
 'housing_woe',
 'contact_woe',
 'month_woe',
 'campaign_woe',
 'response_woe']

In [66]:
X_train_lr_survived = X_train_lr_fin[survived_vars]
X_test_lr_survived = X_test_lr_fin[survived_vars]

In [67]:
logistic_model_2 = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model_2.fit(
    X_train_lr_survived,
    y_train
)

LogisticRegression(max_iter=1000, random_state=42)

In [68]:
logistic_results_2 = evaluate(
    logistic_model_2,
    X_train_lr_survived,
    y_train,
    X_test_lr_survived,
    y_test
)

logistic_results_2

,Dataset,Gini,Precision,Recall
0,Train,51.526737,0.764887,0.944131
1,Test,49.521155,0.782429,0.937190


In [69]:
# For Random Forest

# Use Feature Importance to evaluate which variables are important
# Keep variables with importance between 1-35 %
# Use survived variables to build new model and check performance

In [70]:
feature_importance = pd.DataFrame({
    'Variable': X_train_rf.columns,
    'Importance': rf_model.feature_importances_ * 100
}).sort_values(
    by='Importance',
    ascending=False
)

feature_importance

,Variable,Importance
5,balance,18.831340
0,age,16.697758
9,day,14.044480
10,month,11.499415
1,job,7.624034
11,campaign,6.954637
12,response,6.899556
8,contact,4.556081
3,education,4.085705
2,marital,3.513248


## Importance ≥ 1% və ≤ 35%

In [71]:
survived_rf_vars = feature_importance[
    (feature_importance['Importance'] >= 1) &
    (feature_importance['Importance'] <= 35)
]['Variable'].tolist()

survived_rf_vars

['balance',
 'age',
 'day',
 'month',
 'job',
 'campaign',
 'response',
 'contact',
 'education',
 'marital',
 'housing',
 'loan']

In [72]:
len(survived_rf_vars)

12

In [73]:
X_train_rf_selected = X_train_rf[survived_rf_vars]
X_test_rf_selected = X_test_rf[survived_rf_vars]

In [74]:
rf_model_2 = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf_model_2.fit(
    X_train_rf_selected,
    y_train
)

RandomForestClassifier(n_estimators=300, random_state=42)

In [75]:
rf_results_2 = evaluate(
    rf_model_2,
    X_train_rf_selected,
    y_train,
    X_test_rf_selected,
    y_test
)

rf_results_2

,Dataset,Gini,Precision,Recall
0,Train,100.000000,1.000000,1.000000
1,Test,54.528613,0.809571,0.904132


## Hyperparameter Optimized model

In [76]:
# Optimize paramaters of Random Forest using RandomizedSearchCV

In [77]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

In [78]:
param_grid = {
    'n_estimators': [100, 200, 300, 500, 700],
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['sqrt', 'log2', None],
    'class_weight': [None, 'balanced']
}

In [79]:
rf_base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

In [80]:
rf_random = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_grid,
    n_iter=50,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_random.fit(X_train_rf_selected, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'class_weight': [None, 'balanced'],
                                        'max_depth': [3, 5, 7, 10, 15, None],
                                        'max_features': ['sqrt', 'log2', None],
                                        'min_samples_leaf': [1, 2, 5, 10],
                                        'min_samples_split': [2, 5, 10, 20],
                                        'n_estimators': [100, 200, 300, 500,
                                                         700]},
                   random_state=42, scoring='roc_auc', verbose=1)

In [81]:
rf_random.best_params_

{'n_estimators': 700,
 'min_samples_split': 5,
 'min_samples_leaf': 10,
 'max_features': None,
 'max_depth': None,
 'class_weight': None}

In [82]:
rf_random.best_score_

np.float64(0.7861322118063878)

In [83]:
rf_optimized = rf_random.best_estimator_

In [84]:
# Check model's performance with optimized parameters

In [85]:
rf_optimized_results = evaluate(
    rf_optimized,
    X_train_rf_selected,
    y_train,
    X_test_rf_selected,
    y_test
)

rf_optimized_results

,Dataset,Gini,Precision,Recall
0,Train,85.325731,0.826878,0.942720
1,Test,55.757503,0.810625,0.907989


# WIN MODEL

In [86]:
# Analyze all models performances and write down winning model

In [87]:
# logistic_results → Logistic Initial
# logistic_results_2 → Logistic Survived Variables
# random_forest_results → Random Forest Initial
# rf_results_2 → Random Forest Feature Selection
# rf_optimized_results → Random Forest Feature Selection + Optimization

In [88]:
model_comparison = pd.DataFrame({
    'Model': [
        'Logistic Regression - Initial',
        'Logistic Regression - Survived Variables',
        'Random Forest - Initial',
        'Random Forest - Feature Selection',
        'Random Forest - Optimized'
    ],

    'Train Gini': [
        logistic_results.loc[0, 'Gini'],
        logistic_results_2.loc[0, 'Gini'],
        random_forest_results.loc[0, 'Gini'],
        rf_results_2.loc[0, 'Gini'],
        rf_optimized_results.loc[0, 'Gini']
    ],

    'Test Gini': [
        logistic_results.loc[1, 'Gini'],
        logistic_results_2.loc[1, 'Gini'],
        random_forest_results.loc[1, 'Gini'],
        rf_results_2.loc[1, 'Gini'],
        rf_optimized_results.loc[1, 'Gini']
    ],

    'Gini Gap': [
        abs(logistic_results.loc[0, 'Gini'] - logistic_results.loc[1, 'Gini']),
        abs(logistic_results_2.loc[0, 'Gini'] - logistic_results_2.loc[1, 'Gini']),
        abs(random_forest_results.loc[0, 'Gini'] - random_forest_results.loc[1, 'Gini']),
        abs(rf_results_2.loc[0, 'Gini'] - rf_results_2.loc[1, 'Gini']),
        abs(rf_optimized_results.loc[0, 'Gini'] - rf_optimized_results.loc[1, 'Gini'])
    ],

    'Test Precision': [
        logistic_results.loc[1, 'Precision'],
        logistic_results_2.loc[1, 'Precision'],
        random_forest_results.loc[1, 'Precision'],
        rf_results_2.loc[1, 'Precision'],
        rf_optimized_results.loc[1, 'Precision']
    ],

    'Test Recall': [
        logistic_results.loc[1, 'Recall'],
        logistic_results_2.loc[1, 'Recall'],
        random_forest_results.loc[1, 'Recall'],
        rf_results_2.loc[1, 'Recall'],
        rf_optimized_results.loc[1, 'Recall']
    ]
})


model_comparison.sort_values(
    by='Gini Gap',
    ascending=True
)

,Model,Train Gini,Test Gini,Gini Gap,Test Precision,Test Recall
1,Logistic Regression - Survived Variables,51.526737,49.521155,2.005582,0.782429,0.937190
0,Logistic Regression - Initial,53.115347,50.507083,2.608264,0.786741,0.934986
4,Random Forest - Optimized,85.325731,55.757503,29.568227,0.810625,0.907989
2,Random Forest - Initial,100.000000,55.097435,44.902565,0.809759,0.905234
3,Random Forest - Feature Selection,100.000000,54.528613,45.471387,0.809571,0.904132
